# 35 · Revision-5 statistics on Drive (CPU, ~30–60 min; no training, no evaluation)
Runs the prespecified analysis (protocol_rev5.txt, section C) on all notebook-31 tables, including the frozen cohort and CDD-CESM,
which are too large to download. Read-only for all inputs; writes only `results/p2_rev5/analysis/` (small CSV files).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
IN = '/content/drive/MyDrive/quantum-placement/results/p2_rev5'; OUT = f'{IN}/analysis'
import os; print([f for f in sorted(os.listdir(IN)) if f.startswith('rev5_')])

In [ ]:
src = '# Paper 2, revision 5: prespecified analysis (protocol_rev5.txt, section C) of the notebook-31 per-image tables.\n# Crossed hierarchical bootstrap over seeds and clusters (patients; DMID: source images), B = 10,000, rng(0); Holm within\n# families; equivalence when the 95% CI lies within +-0.02 Dice. Plus seed sensitivity and the additional\n# independent-seed resampling analysis for contrasts whose models do not share the decoder initialization (id, tr).\n# Usage: python3 analysis_rev5.py <dir with rev5_*_per_image.csv.gz> <out dir>\nimport sys, os, numpy as np, pandas as pd\nIN, OUT = sys.argv[1], sys.argv[2]; os.makedirs(OUT, exist_ok=True)\nB = 10_000; EQ = 0.02\nBLOCK = [(\'g_q\', \'g_cl\'), (\'g_q\', \'g_id\'), (\'g_cl\', \'g_id\'), (\'p_q\', \'p_cl\'), (\'p_q\', \'p_id\'), (\'p_cl\', \'p_id\')]\nENC = [(\'p_q\', \'g_q\'), (\'p_cl\', \'g_cl\'), (\'p_id\', \'g_id\')]\nNINE = BLOCK + ENC\nTRIG = [(\'g_q\', \'g_tr\'), (\'p_q\', \'p_tr\')]\nREF = [(\'full\', \'g_id\'), (\'full\', \'p_id\'), (\'unet\', \'full\'), (\'unet\', \'g_q\'), (\'unet\', \'p_q\'), (\'full\', \'g_q\'), (\'full\', \'p_q\')]\nDECODER_DIFFERS = {\'id\', \'tr\'}          # init audit: id and tr controls start from a different decoder initialization\n\n\ndef holm(p):\n    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0\n    for r, i in enumerate(o):\n        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)\n    return adj\n\n\ndef matrix(df, col=\'dice\'):\n    """{model: (seeds, items)} with items aligned; cluster codes. One value per model x seed x crop (mean over fold models)."""\n    g = df.groupby([\'model\', \'seed\', \'roi_id\', \'cluster\'], as_index=False)[col].mean()\n    items = g[[\'roi_id\', \'cluster\']].drop_duplicates(\'roi_id\').sort_values(\'roi_id\').reset_index(drop=True)\n    pos = pd.Series(np.arange(len(items)), index=items.roi_id); codes = pd.factorize(items.cluster)[0]\n    M = {}\n    for (m, s), t in g.groupby([\'model\', \'seed\']):\n        a = np.full(len(items), np.nan); a[pos[t.roi_id].to_numpy()] = t[col].to_numpy(); M.setdefault(m, {})[s] = a\n    return {m: (sorted(v), np.vstack([v[s] for s in sorted(v)])) for m, v in M.items()}, codes\n\n\ndef boot(a, b, codes, rng, joint=True, seeds_a=None, seeds_b=None):\n    ok = ~(np.isnan(a).any(0) | np.isnan(b).any(0)); a, b, c = a[:, ok], b[:, ok], codes[ok]\n    c = pd.factorize(c)[0]; n = c.max() + 1; cnt = np.bincount(c, minlength=n)\n    A = np.vstack([np.bincount(c, r, n) for r in a]); Bm = np.vstack([np.bincount(c, r, n) for r in b])\n    est = a.mean() - b.mean(); out = np.empty(B)\n    for t in range(B):\n        w = np.bincount(rng.integers(0, n, n), minlength=n); den = w @ cnt\n        sa = rng.integers(0, len(A), len(A))\n        sb = sa if (joint and len(A) == len(Bm)) else rng.integers(0, len(Bm), len(Bm))\n        out[t] = (w @ A[sa].mean(0) - w @ Bm[sb].mean(0)) / den\n    p = max(min(1.0, 2 * min((out <= 0).mean(), (out >= 0).mean())), 1 / B)\n    lo, hi = np.percentile(out, [2.5, 97.5]); l9, h9 = np.percentile(out, [5, 95])\n    return dict(diff=est, ci_lo=lo, ci_hi=hi, ci90_lo=l9, ci90_hi=h9, p=p, n_items=int(ok.sum()), n_clusters=int(n),\n                n_seeds=len(A), boot=out)\n\n\ndef verdict(r):\n    if r[\'ci_lo\'] >= -EQ and r[\'ci_hi\'] <= EQ: v = \'equivalent\'\n    else: v = \'inconclusive\'\n    if r[\'ci_lo\'] > 0 or r[\'ci_hi\'] < 0: v = (\'difference, \' + v) if v == \'equivalent\' else \'difference\'\n    return v\n\n\ndef family(M, codes, pairs, rng, label, seeds=None, joint=True):\n    rows = []\n    for a, b in pairs:\n        if a not in M or b not in M: continue\n        sa, Ma = M[a]; sb, Mb = M[b]\n        common = sorted(set(sa) & set(sb)) if seeds is None else [s for s in seeds if s in sa and s in sb]\n        if joint:\n            Ma = Ma[[sa.index(s) for s in common]]; Mb = Mb[[sb.index(s) for s in common]]\n        r = boot(Ma, Mb, codes, rng, joint=joint); r.pop(\'boot\')\n        rows.append(dict(family=label, contrast=f\'{a} - {b}\', A=a, B=b, **r))\n    if rows:\n        adj = holm([r[\'p\'] for r in rows])\n        for r, q in zip(rows, adj): r[\'p_holm\'] = q; r[\'verdict\'] = verdict(r)\n    return rows\n\n\ndef load(name):\n    f = f\'{IN}/rev5_{name}_per_image.csv.gz\'\n    if not os.path.exists(f): print(\'missing\', f); return None\n    d = pd.read_csv(f)\n    d[\'cluster\'] = d[\'patient\'].astype(str) if \'patient\' in d and d[\'patient\'].notna().all() else d.roi_id.str.split(\'_\').str[0]\n    return d\n\n\nrng = np.random.default_rng(0)\nALL = []\nSETS = {\'cbis\': \'CBIS-DDSM test folds\', \'frozen\': \'Frozen CBIS-DDSM cohort\', \'dmid\': \'DMID\', \'csaws\': \'CSAW-S\', \'cddcesm\': \'CDD-CESM\'}\nLEVELS = []\nfor key, title in SETS.items():\n    d = load(key)\n    if d is None: continue\n    refs = [(\'dice\', \'consensus\')] + ([(c, c.replace(\'dice_\', \'\')) for c in d.columns if c.startswith(\'dice_annotator\')] if key == \'csaws\' else [])\n    for col, refname in refs:\n        for sch in (\'train\', \'val\'):\n            ds = d[d.schedule == sch]\n            M, codes = matrix(ds, col)\n            tag = f\'{key}|{refname}|{sch}\'\n            ALL += [dict(set=key, reference=refname, schedule=sch, **r) for r in family(M, codes, NINE, rng, \'nine\')]\n            ALL += [dict(set=key, reference=refname, schedule=sch, **r) for r in family(M, codes, TRIG, rng, \'circuit-trig\')]\n            if sch == \'train\':\n                ALL += [dict(set=key, reference=refname, schedule=sch, **r) for r in family(M, codes, REF, rng, \'reference\')]\n            if refname != \'consensus\': continue\n            # model levels with CI (seeds x clusters)\n            for m, (ss, X) in M.items():\n                zero = np.zeros_like(X); r = boot(X, zero, codes, rng); r.pop(\'boot\')\n                lv = dict(set=key, schedule=sch, model=m, dice=r[\'diff\'], ci_lo=r[\'ci_lo\'], ci_hi=r[\'ci_hi\'], n_seeds=len(ss))\n                sub = ds[ds.model == m]\n                if \'empty\' in sub: lv[\'empty_rate\'] = float(sub.groupby([\'seed\', \'roi_id\']).empty.mean().mean())\n                if \'hd95\' in sub: lv[\'hd95_median\'] = float(sub.groupby([\'seed\', \'roi_id\']).hd95.mean().median())\n                LEVELS.append(lv)\n            if key == \'cbis\':\n                # seed sensitivity: seeds 0-2 only, 3-4 only, block difference, leave one seed out\n                for fam, pairs in ((\'nine\', NINE), (\'circuit-trig\', TRIG)):\n                    for blk, ss in ((\'seeds0-2\', [0, 1, 2]), (\'seeds3-4\', [3, 4])):\n                        ALL += [dict(set=key, reference=refname, schedule=sch, analysis=blk, **r) for r in family(M, codes, pairs, rng, fam, seeds=ss)]\n                    for drop in range(5):\n                        ss = [s for s in range(5) if s != drop]\n                        ALL += [dict(set=key, reference=refname, schedule=sch, analysis=f\'leave-out-seed{drop}\', **r)\n                                for r in family(M, codes, pairs, rng, fam, seeds=ss)]\n                # block difference of each contrast (seeds 0-2 minus 3-4), clusters resampled jointly\n                for a, b in NINE + TRIG:\n                    sa, Ma = M[a]; sb, Mb = M[b]; D = Ma - Mb\n                    x, y = D[:3], D[3:]\n                    c = pd.factorize(codes)[0]; n = c.max() + 1; cnt = np.bincount(c, minlength=n)\n                    X = np.vstack([np.bincount(c, r, n) for r in x]); Y = np.vstack([np.bincount(c, r, n) for r in y]); out = np.empty(B)\n                    for t in range(B):\n                        w = np.bincount(rng.integers(0, n, n), minlength=n)\n                        out[t] = (w @ X[rng.integers(0, 3, 3)].mean(0) - w @ Y[rng.integers(0, 2, 2)].mean(0)) / (w @ cnt)\n                    ALL.append(dict(set=key, reference=refname, schedule=sch, analysis=\'block 0-2 minus 3-4\', family=\'seed-block\',\n                                    contrast=f\'{a} - {b}\', diff=x.mean() - y.mean(), ci_lo=np.percentile(out, 2.5), ci_hi=np.percentile(out, 97.5)))\n                # additional analysis: independent seed resampling where the decoder initialization differs\n                pairs = [(a, b) for a, b in NINE + TRIG if a.split(\'_\')[1] in DECODER_DIFFERS or b.split(\'_\')[1] in DECODER_DIFFERS]\n                ALL += [dict(set=key, reference=refname, schedule=sch, analysis=\'independent seeds (additional)\', **r)\n                        for r in family(M, codes, pairs, rng, \'indep\', joint=False)]\n            print(\'done\', tag, flush=True)\n\nR = pd.DataFrame(ALL); R[\'analysis\'] = R.get(\'analysis\', pd.Series(dtype=str)).fillna(\'main\')\nR.to_csv(f\'{OUT}/rev5_contrasts.csv\', index=False, float_format=\'%.6g\')\nL = pd.DataFrame(LEVELS); L.to_csv(f\'{OUT}/rev5_levels.csv\', index=False, float_format=\'%.6g\')\npd.set_option(\'display.width\', 250)\nshow = [\'set\', \'reference\', \'schedule\', \'family\', \'contrast\', \'diff\', \'ci_lo\', \'ci_hi\', \'p_holm\', \'verdict\']\nprint(R[(R.analysis == \'main\') & (R.reference == \'consensus\')][show].round(4).to_string(index=False))\nprint(L.round(4).to_string(index=False))\n'
open('/content/analysis_rev5.py', 'w').write(src)
print('script written')

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, '/content/analysis_rev5.py', IN, OUT], capture_output=True, text=True)
print(p.stdout[-6000:]); print(p.stderr[-3000:])
print(sorted(os.listdir(OUT))); print('DONE')

In [ ]:
# 4 · Code-replacement drops per image (DMID and CBIS), from every replacement table found; small file for download
import glob, pandas as pd, numpy as np
RES = '/content/drive/MyDrive/quantum-placement/results'
files = sorted(set(glob.glob(f'{RES}/**/*replace*per_image*.csv*', recursive=True)))
print('replacement tables found:'); [print(' ', f) for f in files]
parts = []
for f in files:
    d = pd.read_csv(f)
    if not {'dice', 'dice_const', 'model', 'seed', 'schedule'} <= set(d.columns) or 'dataset' not in d: print('skip', f); continue
    sw = [c for c in d.columns if c.startswith('dice_swap')]
    d['drop_const'] = d.dice - d.dice_const; d['drop_swap'] = d.dice - d[sw].mean(1)
    d['source'] = f.split('/results/')[1]
    parts.append(d[['source', 'dataset', 'roi_id', 'schedule', 'model', 'fold', 'seed', 'dice', 'drop_const', 'drop_swap']])
A = pd.concat(parts, ignore_index=True)
print(A.groupby(['source', 'dataset', 'schedule', 'model']).seed.unique().to_string())
A.to_csv(f'{OUT}/replace_drops_all_sources.csv.gz', index=False, compression='gzip', float_format='%.6g')
print('written', len(A), 'rows | DONE')